### References: https://www.kaggle.com/code/ninamaamary/medqa-eda-embeddings

<div style="background-color: rgba(100, 100, 255, 0.1); padding: 20px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0, 129, 194, 0.5);">
<h1 style="font-size: 26px; font-family: Arial, sans-serif; color: #7e8ff2; text-align: center;"><b>🧬MedQA EDA + Embeddings </b></h1>
</div>

<a id="contents_table"></a>
<div style="background-color: rgba(219, 241, 255, 0.5); padding: 20px; border-radius: 15px; box-shadow: 0 2px 4px 0 rgba(0, 0, 0, 0.1);"> 
     <h1 style="font-size: 24px; font-family: Arial, sans-serif; color: #d9534f;"><b>🥼 🥽 Table of Contents</b></h1>
     
<ul style="font-size:18px; font-family: Arial, sans-serif; color:#7e8ff2;">
        <li><a href="#1" style="color:#333;">Load MedQA Dataset</a></li>
        <li><a href="#2" style="color:#333;">Analyze Dataset</a></li>
        <li><a href="#3" style="color:#333;">Structure and Load Text Books</a></li>
        <ol>
            <li><a href="#31" style="color:#333;">Look at Loaded Books</a></li>
        </ol>
        <li><a href="#4" style="color:#333;">Processing & Chunking Text Books</a></li>
        <ol>
            <li><a href="#41" style="color:#333;">Look at Processed & Chuncked Books</a></li>
            <li><a href="#42" style="color:#333;">Filter Chunks Based on Token Counts</a></li>
        </ol>
        <li><a href="#5" style="color:#333;">Embeddings</a></li>
        <ul>
            <li><a href="#51" style="color:#333;">Save To File</a></li>
        </ul>
    </div>

<a id="1"></a>
# Load MedQA Dataset

In [1]:
import pandas as pd

def load_data(file_path):
    df = pd.read_json(file_path, lines=True)
    return df

In [2]:
TRAIN_PATH = '../datasets/MedQA-USMLE/questions/US/train.jsonl'
VAL_PATH = '../datasets/MedQA-USMLE/questions/US/dev.jsonl'
TEST_PATH = '../datasets/MedQA-USMLE/questions/US/test.jsonl'

In [3]:
train_df = load_data(TRAIN_PATH)
val_df = load_data(VAL_PATH)
test_df = load_data(TEST_PATH)

<a id="2"></a>
# Analyze Dataset

In [4]:
import re
import os
from tqdm import tqdm
    
def count_tokens(text):   
    return len(str(text).split())
    
def combine_options(options_dict):
    return " ".join(options_dict.values())
    
def analyze_medqa_dataset(df):
    # Descriptive Analysis
    total_questions = len(df)
    unique_questions = df['question'].nunique()
    df['question_tokens'] = df['question'].apply(count_tokens)
    total_question_tokens = df['question_tokens'].sum()

    # Distribution of the answers
    correct_answer_distribution = df['answer_idx'].value_counts(normalize=True).mul(100).round(2).to_dict()

    # Tokens Analysis
    df['question_tokens'] = df['question'].apply(count_tokens)
    total_question_tokens = df['question_tokens'].sum()

    df['options_text'] = df['options'].apply(combine_options)
    df['options_tokens'] = df['options_text'].apply(count_tokens)
    total_options_tokens = df['options_tokens'].sum()
    
    # Total tokens
    total_tokens_dataset = total_question_tokens + total_options_tokens

    stats = {
        'Total Questions': total_questions,
        'Unique Questions': unique_questions,
        'Total Question Tokens': total_question_tokens,
        'Total Options Tokens': total_options_tokens,
        'Total Dataset Tokens (Approx.)': total_tokens_dataset, 
        'Average Tokens per Question': (total_question_tokens / total_questions).round(1),
        'Correct Answer Distribution': correct_answer_distribution
    }
    
    return stats
    
def get_medqa_stats(df):
    stats = analyze_medqa_dataset(df)
    display(pd.Series(stats, name='MedQA Dataset Analysis').to_frame())

In [5]:
print('TRAIN: MedQA Analysis')
get_medqa_stats(train_df)

print('VAL: MedQA Analysis')
get_medqa_stats(val_df)

print('TEST: MedQA Analysis')
get_medqa_stats(test_df)

TRAIN: MedQA Analysis


,MedQA Dataset Analysis
Total Questions,10178
Unique Questions,10176
Total Question Tokens,1182460
Total Options Tokens,178922
Total Dataset Tokens (Approx.),1361382
Average Tokens per Question,116.2
Correct Answer Distribution,"{'B': 21.46, 'C': 20.5, 'D': 20.0, 'A': 19.95,..."


VAL: MedQA Analysis


,MedQA Dataset Analysis
Total Questions,1272
Unique Questions,1272
Total Question Tokens,148158
Total Options Tokens,22270
Total Dataset Tokens (Approx.),170428
Average Tokens per Question,116.5
Correct Answer Distribution,"{'B': 21.93, 'D': 21.86, 'A': 20.13, 'C': 19.2..."


TEST: MedQA Analysis


,MedQA Dataset Analysis
Total Questions,1273
Unique Questions,1273
Total Question Tokens,152263
Total Options Tokens,22230
Total Dataset Tokens (Approx.),174493
Average Tokens per Question,119.6
Correct Answer Distribution,"{'B': 21.76, 'A': 21.45, 'D': 21.13, 'C': 19.8..."


<a id="3"></a>
# Structure and Load Text Books

In [6]:
import glob

TEXT_BOOKS = '../datasets/MedQA-USMLE/textbooks/en'

def read_book(file_path):
    paragraphs_list = []
    book = os.path.splitext(os.path.basename(file_path))[0]

    with open(file_path, 'r', encoding='utf-8') as content:
        lines = [line.strip() for line in content.readlines() if line.strip()]

    avg_paragraph_in_page = 4
    idx = 0
    for i in range(0, len(lines), avg_paragraph_in_page):
        paragraph_lines = lines[i:i + avg_paragraph_in_page]
        if paragraph_lines:
            paragraph = "".join(paragraph_lines)

            paragraphs_list.append({
                'book_reference': book,
                'par_id': idx,
                'par_char_count': len(paragraph),
                'par_word_count': len(paragraph.split()),
                'par_token_count': len(paragraph) / 4,
                'text': paragraph
            })
            
            idx += 1

    return paragraphs_list

def read_textbooks(input_directory):
    file_paths = glob.glob(os.path.join(input_directory, '*.txt'))
    books_and_texts = []
    
    for file_path in tqdm(file_paths, desc='Loading files...'):
        books_and_texts.extend(read_book(file_path))

    return books_and_texts
    
def analyze_medqa_books(df):
    display(df.describe().round(2))
    
def get_medqa_books(text_books_path):
    books_and_texts = read_textbooks(text_books_path)
    df = pd.DataFrame(books_and_texts)
    
    print('MedQA Text Books Analysis:')
    analyze_medqa_books(df)
    
    return df, books_and_texts

In [7]:
text_book_df, books_and_texts = get_medqa_books(TEXT_BOOKS)

Loading files...: 100%|██████████| 18/18 [00:00<00:00, 20.93it/s]


MedQA Text Books Analysis:


,par_id,par_char_count,par_word_count,par_token_count
count,53340.00,53340.00,53340.00,53340.00
mean,2333.78,1663.02,237.94,415.76
std,2149.63,11951.03,1677.31,2987.76
min,0.00,8.00,1.00,2.00
25%,782.00,496.00,67.00,124.00
50%,1655.00,1120.00,157.00,280.00
75%,3147.00,2174.00,315.25,543.50
max,9880.00,1163314.00,163280.00,290828.50


<a id="31"></a>
## Look at Loaded Books

In [8]:
books_and_texts[-1]

{'book_reference': 'Surgery_Schwartz',
 'par_id': 31,
 'par_char_count': 44064,
 'par_word_count': 6036,
 'par_token_count': 11016.0,
 'text': 'Brunicardi_Ch53_p2163-p2186.indd   218622/02/19   4:39 PMThis page intentionally left blankWeb-Based Education and Implications of Social MediaLillian S. Kao and Michael E. Zenilman 54chapterINTRODUCTIONSurgical education has changed significantly over the past two decades. Disruptive forces such as work hour restrictions and the advent of laparoscopy have forced educators to rethink how and where to teach residents. Technologies, including the inter-net and web-based applications, have further enabled educators to redesign surgical education (Fig. 54-1). The internet has become an integral tool not just in surgical education but also in Americans’ lives by changing the way that people communicate with each other, access information, and conduct their daily lives. Today, almost 9 in 10 American adults use the internet. Furthermore, the internet

<a id="4"></a>
# Preprocessing & Chunking Text Books 

In [9]:
from spacy.lang.en import English

nlp = English()
nlp.max_length = 100_000_000
nlp.add_pipe('sentencizer')


def clean_for_sentencizer(text):
    # Mask decimal
    text = re.sub(r'(\d)\.(\d)', r'\1<D>\2', text)
    
    # Mask common medicle abbreviations followed by a period (e.g., eFig., Fig., Table., Dr., i.e., e.g.).
    text = re.sub(r'(eFig|Fig|Table|Dr|Mr|Mrs|Ms|i\.e|e\.g|etc)\.', r'\1<P>', text)
    return text
    
def split_list(input_list, slice_size):
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]


def process_and_chunk(paragraphs, nlp_model, num_sentence_chunk_size=10):
    final_chunks = []
    
    for item in tqdm(paragraphs, desc='Processing & Chunking...'):
        cleaned_text = clean_for_sentencizer(item['text'])
        doc = nlp_model(cleaned_text)
    
        sentences = [str(sentence).replace('<D>', '.').replace('<P>', '.') for sentence in doc.sents if str(sentence).strip()]
        sentence_chunks = split_list(input_list=sentences, slice_size=num_sentence_chunk_size)
        
        for chunk_idx, sentence_chunk in enumerate(sentence_chunks):
            joined_chunk = " ".join(sentence_chunk).strip()
            chunk_doc = nlp_model(joined_chunk)
            
            final_chunks.append({
                'book_reference': item['book_reference'],
                'chunk_id': f"{item['book_reference']}_{item['par_id']}_{chunk_idx}",
                'chunk_text': joined_chunk,
                'chunk_size_sentences': len(sentence_chunk),
                'chunk_char_count': len(joined_chunk),
                'chunk_token_count': len(chunk_doc),
                'chunk_word_count': len([token for token in chunk_doc if token.is_alpha]),
            })
            
    return final_chunks

final_chunks = process_and_chunk(books_and_texts, nlp, num_sentence_chunk_size=10)

Processing & Chunking...: 100%|██████████| 53340/53340 [01:47<00:00, 494.56it/s] 


<a id="41"></a>
## Look at Processed & Chuncked Books

In [10]:
df = pd.DataFrame(final_chunks)
df.describe().round(2)

,chunk_size_sentences,chunk_char_count,chunk_token_count,chunk_word_count
count,91756.00,91756.00,91756.00,91756.00
mean,7.07,967.46,168.41,135.62
std,3.22,596.15,102.67,87.14
min,1.00,2.00,1.00,0.00
25%,4.00,442.00,81.00,57.00
50%,8.00,943.00,163.00,131.00
75%,10.00,1399.00,243.00,201.00
max,10.00,10405.00,1542.00,977.00


In [11]:
final_chunks[-1]

{'book_reference': 'Surgery_Schwartz',
 'chunk_id': 'Surgery_Schwartz_31_29',
 'chunk_text': 'Surgical education in the internet era. J Surg Res. 2009;156(2):177-182. This article describes the factors that led to a change in surgical education over the last two decades. Taveira-Gomes T, Ferreira P, Taveira-Gomes I, Severo M, Ferreira MA. What are we looking for in computer-based learning inter-ventions in medical education? A systematic review. J Med Internet Res. 2016;18(8):e204. This systematic review assessed recent studies on computer-based learning (CBL) for types of software platforms and interventions and adherence to current recommendations for CBL research.Brunicardi_Ch54_p2187-p2196.indd   219513/02/19   2:37 PM',
 'chunk_size_sentences': 10,
 'chunk_char_count': 638,
 'chunk_token_count': 115,
 'chunk_word_count': 86}

<a id="42"></a>
## Filter Chunks Based on Token Counts

In [12]:
min_token_length = 30
pages_and_chunks_over_min_token_len = df[df['chunk_token_count'] > min_token_length].to_dict(orient='records')

len(pages_and_chunks_over_min_token_len)

86236

<a id="5"></a>
# Embeddings

In [13]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Runing on: {device}")

Runing on: cuda


In [14]:
%%capture
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=device)

In [15]:
text_chunks = [item['chunk_text'] for item in pages_and_chunks_over_min_token_len] 


text_chunk_embeddings = embedding_model.encode(
    text_chunks, 
    batch_size=32, 
    convert_to_tensor=True, 
    show_progress_bar=True
)

Batches: 100%|██████████| 2695/2695 [10:19<00:00,  4.35it/s]


In [16]:
for i in range(len(pages_and_chunks_over_min_token_len)):
    pages_and_chunks_over_min_token_len[i]['embedding'] = text_chunk_embeddings[i].tolist()

In [17]:
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
text_chunks_and_embeddings_df.head()

,book_reference,chunk_id,chunk_text,chunk_size_sentences,chunk_char_count,chunk_token_count,chunk_word_count,embedding
0,Anatomy_Gray,Anatomy_Gray_0_0,What is anatomy?Anatomy includes those structu...,10,1308,226,199,"[0.018424157053232193, 0.003207121742889285, -..."
1,Anatomy_Gray,Anatomy_Gray_0_1,Knowing the names of the various branches of t...,3,562,102,96,"[0.03533714637160301, 0.007124538533389568, -0..."
2,Anatomy_Gray,Anatomy_Gray_1_0,How can gross anatomy be studied?The term anat...,5,703,136,114,"[0.05197994038462639, -0.0040394761599600315, ..."
3,Anatomy_Gray,Anatomy_Gray_2_0,"This includes the vasculature, the nerves, the...",10,1336,252,210,"[0.017961757257580757, -0.00828748568892479, -..."
4,Anatomy_Gray,Anatomy_Gray_3_0,The anatomical position is the standard refere...,9,1098,210,190,"[0.010828881524503231, -0.06143888086080551, -..."


<a id="51"></a>
## Save To File

In [18]:
# Save embeddings to file
embeddings_df_save_path = 'text_chunks_and_embeddings.parquet'
text_chunks_and_embeddings_df.to_parquet(embeddings_df_save_path)